#Init

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import StringType,DateType
from pyspark.sql.functions import trim,col,length

#Reading From Bronze Layer

In [0]:
df=spark.table("workspace.bronze.erp_cust_az12")

#Data Transformation

#Trimming

In [0]:
for field in df.schema.fields:
    if isinstance(field.dataType, StringType):
        df = df.withColumn(field.name, trim(col(field.name)))

#Normalization

## Customer Id Cleanup

In [0]:

df = df.withColumn(
    "cid",
    F.when(col("cid").startswith("NAS"),
           F.substring(col("cid"), 4, F.length(col("cid"))))
     .otherwise(col("cid"))
)


## Birth date Validation

In [0]:
df=df.withColumn(
    'BDATE',
    F.when(col('BDATE')>F.current_date(),None)
    .otherwise(col('BDATE')
))


## Gender Cleaning

In [0]:
df= df.withColumn(
    "gen",
    F.when(F.upper(col('gen')).isin('M','MALE'),'Male')
    .when(F.upper(col('gen')).isin('F','FEMALE'),'Female')
    .otherwise('n/a')
)

#Renaming columns

In [0]:
rename_cols = {
    "cid": "customer_number",
    "bdate": "birth_date",
    "gen": "gender"
}

for old_name, new_name in rename_cols.items():
    df = df.withColumnRenamed(old_name, new_name)


## Sanity checks of dataframe

In [0]:
display(df.limit(10))

#Write to Silver Layer

In [0]:
(
df.write
.mode("overwrite")
.format('delta')
.saveAsTable("silver.erp_customers")
)